# Task 1 · MLP vs CNN
Every run trains on 4 folds of the training set (40,000 images) and validates on the 5th (10,000).
Numbers are mean (SD) over the 5 folds; runs are cached in `out/runs`.

In [ ]:
from hw3 import *

X, Y, Xte, Yte = cifar()
X.shape, Xte.shape, [len(f) for f in folds()], Y[folds()[0]].bincount().tolist()

## Hyperparameter search

In [ ]:
LRS = (3e-4, 1e-3, 3e-3)
SIZES = {'mlp': [(512, 256), (1024, 512, 256)], 'cnn': [16, 32]}
widths = lambda m, s: '-'.join(map(str, s if m == 'mlp' else (s, 2 * s, 4 * s)))
cfgs = {f'{m}_{widths(m, s)}_{lr:g}': dict(model=partial(MLP if m == 'mlp' else CNN, s), lr=lr)
        for m in SIZES for s in SIZES[m] for lr in LRS}
R = cv(cfgs)

In [ ]:
hp = pd.DataFrame([k.split('_') + [f"{R[k][0]['params'] / 1e6:.2f}", col(R[k], 'val_acc').mean(),
                                   ms(col(R[k], 'val_acc')), ms(col(R[k], 'val_loss'), 1, 3),
                                   ms([r['time'].mean() for r in R[k]], 1, 0)] for k in R],
                  index=list(R), columns=['model', 'widths', 'max lr', 'params (M)', 'acc', 'val acc', 'val loss', 's/epoch'])
best = hp.groupby('model').acc.idxmax()
json.dump({m: dict(key=k, arch=cfgs[k]['model'].args[0], lr=cfgs[k]['lr']) for m, k in best.items()},
          open(os.path.join(OUT, 'best.json'), 'w'))
table = hp.drop(columns='acc').assign(model=hp.model.str.upper())
with open(os.path.join(OUT, 'hp.tex'), 'w') as f:
    f.writelines(' & '.join(r) + ' \\\\\n' for r in table.astype(str).values)
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.4))
for ax, m in zip(axes, SIZES):
    for s, a in zip(SIZES[m], (.5, 0)):
        v = np.array([col(R[f'{m}_{widths(m, s)}_{lr:g}'], 'val_acc') for lr in LRS]) * 100
        ax.errorbar(LRS, v.mean(1), v.std(1, ddof=1), marker='o', capsize=3, color=mix(COL[m], 'white', a), label=widths(m, s))
    ax.set(xscale='log', xlabel='max learning rate', ylabel='CV accuracy (%)', title=m.upper())
    ax.legend(title='layer widths')
save(fig, 'hp')

## Selected architectures

In [ ]:
i = int((Y == 7).nonzero()[0])
mlp, cnn = (cfgs[best[m]]['model'] for m in ('mlp', 'cnn'))
draw([(3072, 'flatten', MUTED)] + [(h, 'FC + ReLU', COL['mlp']) for h in mlp.args[0]] + [(10, 'softmax', INK)], X[i])
save(plt.gcf(), 'arch_mlp')
draw(cnn_layers(cnn.args[0]), X[i])
save(plt.gcf(), 'arch_cnn')

## Accuracy, size, speed and memory

In [ ]:
P = pd.DataFrame({m: profile(cfgs[k]['model']) for m, k in best.items()}).T
P['val'], P['test'] = ([ms(col(R[k], key)) for k in best] for key in ('val_acc', 'test_acc'))
P

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.3))
curves({m.upper(): (R[k], COL[m]) for m, k in best.items()}, axes[:2])
for m, k in best.items():
    v = np.stack([np.asarray(r['val_acc']) for r in R[k]]).mean(0) * 100
    axes[2].plot(P.epoch[m] / 60 * np.arange(1, len(v) + 1), v, 'o-', ms=3, color=COL[m], label=m.upper())
axes[2].set(xlabel='training time (min, 4 threads)', ylabel='val accuracy (%)')
save(fig, 'curves')

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(7.2, 1.9))
names, colors = [m.upper() for m in best.index], [COL[m] for m in best.index]
acc = np.array([col(R[k], 'test_acc') for k in best]) * 100
axes[0].bar(names, acc.mean(1), yerr=acc.std(1, ddof=1), capsize=3, color=colors)
for ax, key, k in zip(axes[1:4], ('params', 'mflops', 'epoch'), (1e6, 1, 1)):
    ax.bar(names, P[key] / k, color=colors)
for j, m in enumerate(best.index):
    axes[4].bar(np.arange(3) + .4 * j - .2, P.loc[m, ['weights', 'adam', 'acts']].astype(float), .38, color=COL[m])
axes[4].set_xticks(range(3), ['weights', 'grad+Adam', 'activ.'], rotation=30)
for ax, t in zip(axes, ('test accuracy (%)', 'parameters (M)', 'MFLOPs / image', 's / epoch', 'training MB')):
    ax.set_title(t)
save(fig, 'compare')

## What the weights look like

In [ ]:
mlp, cnn = (load(best[m], cfgs[best[m]]['model']) for m in ('mlp', 'cnn'))
W = mlp[1].weight.detach()
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.1), gridspec_kw=dict(width_ratios=[2, 1]))
tiles(axes[0], W[W.norm(dim=1).argsort(descending=True)[:32]].view(-1, 3, 32, 32))
tiles(axes[1], cnn[0][0].weight.detach())
axes[0].set_title('MLP: weights of 32 first-layer units (32×32×3 each)')
axes[1].set_title(f'CNN: all {len(cnn[0][0].weight)} first-layer filters (3×3×3)')
save(fig, 'weights')

In [ ]:
j = int((Yte == 7).nonzero()[0])
x, maps = prep(Xte[j:j + 1]), []
with torch.no_grad():
    for b in cnn[:3]:
        a = b[:3](x)[0]
        maps.append(a[a.mean((1, 2)).argsort(descending=True)[:8], None])
        x = b(x)
fig, axes = plt.subplots(1, 4, figsize=(7.2, 1.6), gridspec_kw=dict(width_ratios=[1, 2, 2, 2]))
axes[0].imshow(Xte[j].permute(1, 2, 0))
axes[0].axis('off')
for r, (ax, a) in enumerate(zip(axes[1:], maps)):
    tiles(ax, a, 4)
    ax.set_title(f'conv {r + 1}: 8 most active maps ({a.shape[-1]}×{a.shape[-1]})')
save(fig, 'featuremaps')

## Shuffling the pixels
One fixed random permutation of the 1,024 pixel positions, applied to every image: the MLP cannot tell, the CNN loses
the neighbourhoods its 3×3 filters rely on.

In [ ]:
Q = cv({f'{k}_perm': cfgs[k] | dict(kind='perm') for k in best})
x = Xte[j:j + 1]
acc = np.array([[col(R[k], 'test_acc'), col(Q[f'{k}_perm'], 'test_acc')] for k in best]) * 100
fig, axes = plt.subplots(1, 3, figsize=(7.2, 1.9), gridspec_kw=dict(width_ratios=[1, 1, 2.6]))
for ax, img, t in zip(axes, (x, x.flatten(2)[..., PERM].view_as(x)), ('original', 'shuffled pixels')):
    ax.imshow(img[0].permute(1, 2, 0))
    ax.set_title(t)
    ax.axis('off')
for c, (cond, hatch) in enumerate((('original', None), ('shuffled pixels', '///'))):
    axes[2].bar(np.arange(2) + .4 * c - .2, acc[:, c].mean(1), .38, yerr=acc[:, c].std(1, ddof=1), capsize=3, hatch=hatch,
                color=[mix(COL[m], 'white', .45 * c) for m in best.index], edgecolor='white', label=cond)
axes[2].set(xticks=range(2), xticklabels=[m.upper() for m in best.index], ylabel='test accuracy (%)')
axes[2].legend(loc='lower right')
save(fig, 'perm')

In [ ]:
tex('t1', **{m + n: v for m, k in best.items() for n, v in dict(
    Widths=hp.widths[k], LR=hp['max lr'][k], Val=ms(col(R[k], 'val_acc')), Test=ms(col(R[k], 'test_acc')),
    Perm=ms(col(Q[f'{k}_perm'], 'test_acc')), Params=f'{P.params[m] / 1e6:.2f}', Flops=f'{P.mflops[m]:.1f}',
    Epoch=f'{P.epoch[m]:.0f}', Mem=f"{P.loc[m, ['weights', 'adam', 'acts']].sum():.0f}").items()})